# Build Your Own Copilot in Pure Python

Companion notebook for the blog post at [cmenguy.github.io](https://cmenguy.github.io).

This notebook walks through building a minimal AI code completion tool:
1. Fill-in-the-Middle (FIM) inference with a code LLM
2. An editor-like completion interface
3. An LSP-style HTTP completion server
4. Fine-tuning on a custom codebase with LoRA
5. Evaluation of completion quality

## Setup

Install required packages. This notebook uses `tiny_starcoder_py` (164M params) which runs on CPU or a free Colab GPU.

In [ ]:
!pip install -q transformers torch peft trl datasets accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "bigcode/tiny_starcoder_py"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

## Fill-in-the-Middle (FIM) Inference

FIM lets the model generate code given both what comes *before* and *after* the cursor.
Three special tokens mark the boundaries: `<fim_prefix>`, `<fim_suffix>`, `<fim_middle>`.

In [ ]:
FIM_PREFIX = "<fim_prefix>"
FIM_SUFFIX = "<fim_suffix>"
FIM_MIDDLE = "<fim_middle>"


def fim_complete(prefix, suffix, max_new_tokens=64,
                 temperature=0.2, top_p=0.95, _model=None, _tokenizer=None):
    """Generate code to fill between prefix and suffix."""
    _model = _model or model
    _tokenizer = _tokenizer or tokenizer

    prompt = f"{FIM_PREFIX}{prefix}{FIM_SUFFIX}{suffix}{FIM_MIDDLE}"
    inputs = _tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = _model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=_tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    generated = outputs[0, inputs["input_ids"].shape[1]:]
    completion = _tokenizer.decode(generated, skip_special_tokens=True)

    # Stop at end-of-middle or end-of-text token
    for stop_token in ["<fim_middle>", "<|endoftext|>"]:
        if stop_token in completion:
            completion = completion[:completion.index(stop_token)]

    return completion.rstrip()

### Test FIM: Binary Search

The cursor is between the `while` loop body setup and the `elif` branch.
The model should generate the `if arr[mid] == target: return mid` condition.

In [ ]:
prefix = """def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        """

suffix = """
        elif arr[mid] > target:
            right = mid - 1
        else:
            left = mid + 1
    return -1
"""

result = fim_complete(prefix, suffix)
print("Generated:")
print(result)

## Left-to-Right Completion

When the cursor is at the end of the file (no code below), standard left-to-right completion is used.

In [ ]:
def complete_code(code, max_new_tokens=128,
                  temperature=0.2, top_p=0.95):
    """Standard left-to-right code completion."""
    inputs = tokenizer(code, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0, inputs["input_ids"].shape[1]:]
    completion = tokenizer.decode(generated, skip_special_tokens=True)

    if "<|endoftext|>" in completion:
        completion = completion[:completion.index("<|endoftext|>")]

    return completion.rstrip()

In [ ]:
code = """import json

def parse_config(filepath):
    \"\"\"Read a JSON config file and return a dict.\"\"\"
"""

result = complete_code(code)
print("Generated:")
print(result)

## Editor-Like Interface

A real Copilot integration decides whether to use FIM or left-to-right based on the cursor position.

In [ ]:
def get_completion(document, cursor_line, cursor_col):
    """
    Given a document (list of lines), cursor position,
    decide whether to use FIM or left-to-right completion.
    """
    lines = document.split("\n") if isinstance(document, str) else document

    # Split at cursor position
    prefix_lines = lines[:cursor_line]
    current_line = lines[cursor_line][:cursor_col]
    suffix_lines = lines[cursor_line + 1:]

    prefix = "\n".join(prefix_lines + [current_line])
    suffix = "\n".join(suffix_lines)

    # If there's meaningful code below, use FIM
    has_suffix = any(line.strip() for line in suffix_lines)

    if has_suffix:
        return fim_complete(prefix, suffix)
    else:
        return complete_code(prefix)

In [ ]:
document = """class Calculator:
    def __init__(self):
        self.history = []

    def add(self, a, b):
        result = a + b
        self.history.append(('add', a, b, result))
        return result

    def subtract(self, a, b):

    def get_history(self):
        return self.history
"""

# Cursor is inside the empty subtract method (line 10, col 0)
completion = get_completion(document, cursor_line=10, cursor_col=0)
print("FIM completion for subtract():")
print(completion)

## HTTP Completion Server

A minimal server that any editor with an HTTP plugin can talk to.

In [ ]:
from http.server import HTTPServer, BaseHTTPRequestHandler
import json as json_mod
import threading


class CompletionHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        content_length = int(self.headers["Content-Length"])
        body = json_mod.loads(self.rfile.read(content_length))

        document = body["document"]
        line = body["line"]
        col = body["col"]

        completion = get_completion(document, line, col)

        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        response = json_mod.dumps({"completion": completion})
        self.wfile.write(response.encode())

    def log_message(self, format, *args):
        pass  # Suppress default logging


def start_server(port=8765):
    server = HTTPServer(("localhost", port), CompletionHandler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    print(f"Completion server running on http://localhost:{port}")
    return server


# Uncomment to start the server:
# server = start_server(8765)

## Context Gathering

Production tools gather context from imports, open tabs, and the wider repo.
Here's a simplified version that extracts signatures from imported modules.

In [ ]:
import ast
import os


def extract_signatures(filepath):
    """Pull function/class signatures from a Python file."""
    with open(filepath) as f:
        try:
            tree = ast.parse(f.read())
        except SyntaxError:
            return ""

    sigs = []
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            arg_names = [a.arg for a in node.args.args]
            sig = f"def {node.name}({', '.join(arg_names)}): ..."
            if (node.body and isinstance(node.body[0], ast.Expr)
                    and isinstance(node.body[0].value, ast.Constant)):
                docstring = node.body[0].value.value
                first_line = docstring.strip().split("\n")[0]
                sig += f'  # {first_line}'
            sigs.append(sig)
        elif isinstance(node, ast.ClassDef):
            sigs.append(f"class {node.name}: ...")

    return "\n".join(sigs)


def gather_context(filepath, project_root, max_context_tokens=2048):
    """
    Collect relevant context beyond the current file.
    """
    with open(filepath) as f:
        source = f.read()

    context_parts = []

    try:
        tree = ast.parse(source)
    except SyntaxError:
        return ""

    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom) and node.module:
            module_path = os.path.join(
                project_root,
                node.module.replace(".", "/") + ".py"
            )
            if os.path.exists(module_path):
                sigs = extract_signatures(module_path)
                if sigs:
                    context_parts.append(
                        f"# From {node.module}:\n{sigs}"
                    )

    return "\n\n".join(context_parts)

### Test: Extract signatures from a sample file

In [ ]:
# Create a sample file to test signature extraction
sample_code = '''
class DataProcessor:
    """Processes raw data for ML pipelines."""

    def __init__(self, config):
        """Initialize with a config dict."""
        self.config = config

    def transform(self, data, columns):
        """Apply transformations to specified columns."""
        pass

    def validate(self, data):
        """Check data quality constraints."""
        pass


def load_dataset(path, format="parquet"):
    """Load a dataset from disk."""
    pass
'''

os.makedirs("/tmp/sample_project", exist_ok=True)
with open("/tmp/sample_project/data_utils.py", "w") as f:
    f.write(sample_code)

sigs = extract_signatures("/tmp/sample_project/data_utils.py")
print(sigs)

## Post-Processing Completions

In [ ]:
def postprocess_completion(completion, indent_level,
                           existing_lines=None):
    """
    Clean up a raw model completion for display.
    """
    lines = completion.split("\n")
    result = []

    for line in lines:
        # Stop at a line that starts a new top-level definition
        stripped = line.lstrip()
        if stripped.startswith(("class ", "def ")) and not line.startswith(" "):
            break

        # Fix indent: ensure consistent with surrounding code
        if line.strip():
            result.append(" " * indent_level + line.lstrip())
        else:
            result.append("")

    # Remove trailing empty lines
    while result and not result[-1].strip():
        result.pop()

    # Deduplicate against existing code
    if existing_lines:
        while (result and existing_lines
               and result[-1].strip() == existing_lines[0].strip()):
            result.pop()

    return "\n".join(result)


# Test post-processing
raw = """        result = a - b
        self.history.append(('subtract', a, b, result))
        return result

    def get_history(self):"""

cleaned = postprocess_completion(
    raw, indent_level=8,
    existing_lines=["    def get_history(self):"]
)
print("Post-processed:")
print(cleaned)

## Fine-Tuning on a Custom Codebase

### Step 1: Prepare Training Data

Collect Python files, filter for quality, and create FIM training examples.

In [ ]:
import random

random.seed(42)


def collect_python_files(repo_path, max_file_size=50000):
    """Walk a repo and collect Python files."""
    files = []
    skip_dirs = {
        ".git", "__pycache__", "node_modules",
        ".venv", "venv", ".tox", "build", "dist",
        ".eggs",
    }

    for root, dirs, filenames in os.walk(repo_path):
        dirs[:] = [
            d for d in dirs
            if d not in skip_dirs and not d.startswith(".")
        ]

        for fname in filenames:
            if not fname.endswith(".py"):
                continue
            filepath = os.path.join(root, fname)
            size = os.path.getsize(filepath)
            if size > max_file_size or size < 50:
                continue
            files.append(filepath)

    return files


def read_and_filter(filepath):
    """Read a file and apply quality filters."""
    with open(filepath, encoding="utf-8", errors="ignore") as f:
        content = f.read()

    first_line = content.split("\n")[0] if content else ""
    if "auto-generated" in first_line.lower():
        return None
    if "do not edit" in first_line.lower():
        return None

    lines = content.split("\n")
    code_lines = [l for l in lines if l.strip() and not l.strip().startswith("#")]
    if len(code_lines) < 5:
        return None

    return content

In [ ]:
def create_fim_examples(content, num_splits=3):
    """
    Create FIM training examples from a single file.
    Split at random points to create prefix/middle/suffix triples.
    """
    lines = content.split("\n")
    if len(lines) < 5:
        return []

    examples = []
    for _ in range(num_splits):
        split_start = random.randint(1, max(1, len(lines) - 3))
        mid_len = random.randint(1, min(5, len(lines) - split_start))
        split_end = split_start + mid_len

        prefix = "\n".join(lines[:split_start])
        middle = "\n".join(lines[split_start:split_end])
        suffix = "\n".join(lines[split_end:])

        if not middle.strip():
            continue

        fim_text = (
            f"<fim_prefix>{prefix}\n"
            f"<fim_suffix>\n{suffix}"
            f"<fim_middle>{middle}"
        )
        examples.append(fim_text)

    return examples

In [ ]:
def build_dataset(repo_path, output_path="train_data.jsonl"):
    """Build a JSONL training dataset from a repo."""
    import json

    files = collect_python_files(repo_path)
    print(f"Found {len(files)} Python files")

    examples = []
    for filepath in files:
        content = read_and_filter(filepath)
        if content is None:
            continue
        file_examples = create_fim_examples(content)
        examples.extend(file_examples)

    random.shuffle(examples)

    with open(output_path, "w") as f:
        for ex in examples:
            f.write(json.dumps({"text": ex}) + "\n")

    print(f"Created {len(examples)} training examples")
    return examples

### Demo: Create FIM examples from sample code

In [ ]:
# Demo with the sample code we created earlier
demo_examples = create_fim_examples(sample_code, num_splits=2)
for i, ex in enumerate(demo_examples):
    print(f"--- Example {i+1} ---")
    print(ex[:300])
    print("...")
    print()

### Step 2: LoRA Fine-Tuning

We use LoRA to train <1% of the model's parameters on our codebase.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "c_attn",   # attention QKV projection
        "c_proj",   # attention output projection
        "c_fc",     # MLP first layer
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

To actually run training, you'd need a prepared dataset from `build_dataset()` above.
Here's the training config (uncomment to run with your own data):

In [ ]:
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

# To train on your own repo:
# 1. Run: build_dataset("/path/to/your/repo", "train_data.jsonl")
# 2. Uncomment the code below:

# dataset = load_dataset(
#     "json", data_files="train_data.jsonl", split="train"
# )
#
# training_config = SFTConfig(
#     output_dir="./copilot-finetuned",
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.1,
#     logging_steps=10,
#     save_strategy="epoch",
#     bf16=True,
#     max_seq_length=1024,
#     dataset_text_field="text",
# )
#
# trainer = SFTTrainer(
#     model=peft_model,
#     args=training_config,
#     train_dataset=dataset,
#     processing_class=tokenizer,
# )
#
# trainer.train()

print("Training config ready. Uncomment above to train on your own codebase.")

### Loading a Fine-Tuned Model

After training, load the LoRA adapter and merge it into the base model for production inference.

In [ ]:
from peft import PeftModel

# After training, you would load like this:
#
# base_model = AutoModelForCausalLM.from_pretrained(model_name)
# finetuned_model = PeftModel.from_pretrained(
#     base_model, "./copilot-finetuned/checkpoint-final"
# )
#
# # Merge LoRA weights into base for zero-overhead inference
# merged_model = finetuned_model.merge_and_unload()

print("Load pattern ready. After training, uncomment to load fine-tuned model.")

### Multi-Repo Adapters

When you have multiple repos with different conventions, train a separate LoRA adapter per repo and swap them at runtime. Each adapter is small (~14MB for this model, ~50-100MB for a production 15B model).

In [ ]:
import tempfile

# Demo: create two fake adapters and show multi-adapter loading/switching
base_for_demo = AutoModelForCausalLM.from_pretrained(model_name)

demo_lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

# Save two adapters (simulating two different repos)
adapter_a = get_peft_model(
    AutoModelForCausalLM.from_pretrained(model_name), demo_lora_config
)
tmpdir_a = tempfile.mkdtemp()
adapter_a.save_pretrained(tmpdir_a)

adapter_b = get_peft_model(
    AutoModelForCausalLM.from_pretrained(model_name), demo_lora_config
)
tmpdir_b = tempfile.mkdtemp()
adapter_b.save_pretrained(tmpdir_b)

print(f"Adapter A size: {sum(os.path.getsize(os.path.join(tmpdir_a, f)) for f in os.listdir(tmpdir_a) if os.path.isfile(os.path.join(tmpdir_a, f))) / 1024 / 1024:.1f} MB")

# Load both adapters onto one base model
multi_model = PeftModel.from_pretrained(
    base_for_demo, tmpdir_a, adapter_name="data_eng"
)
multi_model.load_adapter(tmpdir_b, adapter_name="ml_training")

# Switch between them at runtime
multi_model.set_adapter("data_eng")
print("Active adapter: data_eng")

multi_model.set_adapter("ml_training")
print("Active adapter: ml_training")

# Weighted merge with TIES-Merging
multi_model.add_weighted_adapter(
    adapters=["data_eng", "ml_training"],
    weights=[0.7, 0.3],
    adapter_name="combined",
    combination_type="ties",
    density=0.5,
)
multi_model.set_adapter("combined")
print("Active adapter: combined (TIES-Merged, 70/30)")
print("\nAll multi-adapter operations work!")

## Evaluation

### Edit Distance

In [ ]:
def edit_distance(s1, s2):
    """Levenshtein distance between two strings."""
    if len(s1) < len(s2):
        return edit_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)

    prev_row = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = prev_row[j + 1] + 1
            deletions = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row

    return prev_row[-1]


# Quick test
print(f"edit_distance('return result', 'return result'): {edit_distance('return result', 'return result')}")
print(f"edit_distance('return result', 'return res'): {edit_distance('return result', 'return res')}")
print(f"edit_distance('self.data', 'self.items'): {edit_distance('self.data', 'self.items')}")

### Held-Out File Evaluation

In [ ]:
def eval_completion_accuracy(test_files, num_samples=100):
    """
    For each test file, mask a random chunk and measure
    whether the model's completion matches the original.
    """
    exact_matches = 0
    edit_distances = []

    for filepath in test_files[:num_samples]:
        content = read_and_filter(filepath)
        if content is None:
            continue

        lines = content.split("\n")
        if len(lines) < 5:
            continue

        mask_line = random.randint(1, len(lines) - 2)
        prefix = "\n".join(lines[:mask_line])
        expected = lines[mask_line]
        suffix = "\n".join(lines[mask_line + 1:])

        generated = fim_complete(prefix, suffix, max_new_tokens=64)
        first_line = generated.split("\n")[0]

        if first_line.strip() == expected.strip():
            exact_matches += 1

        ed = edit_distance(first_line.strip(), expected.strip())
        edit_distances.append(ed)

    evaluated = min(num_samples, len(test_files))
    if evaluated == 0:
        return {"exact_match": 0, "avg_edit_distance": 0}

    accuracy = exact_matches / evaluated
    avg_ed = sum(edit_distances) / len(edit_distances) if edit_distances else 0
    return {
        "exact_match": accuracy,
        "avg_edit_distance": avg_ed,
    }


# To evaluate on your repo:
# test_files = collect_python_files("/path/to/repo")
# random.shuffle(test_files)
# results = eval_completion_accuracy(test_files[:20])
# print(results)

print("Evaluation function ready. Point it at your repo to measure quality.")

## Summary

We built a working code completion pipeline:

1. **FIM inference** using a code LLM with prefix/suffix/middle format
2. **Completion routing** that decides between FIM and left-to-right based on cursor position
3. **An HTTP server** that any editor can talk to
4. **Context gathering** from imports and project structure
5. **Fine-tuning pipeline** with LoRA on your own codebase
6. **Evaluation** with exact match and edit distance metrics

The full blog post at [cmenguy.github.io](https://cmenguy.github.io) covers the tradeoffs between RAG, LoRA, and full fine-tuning, plus the production architecture of real Copilot tools.